# FaceRestore Training Pipeline

**Setup:** Runtime → Change runtime type → **T4 GPU**

**Upload these 3 files to Google Drive root:**
1. `inpaint_training.zip` — training code
2. `celeba_256.zip` — preprocessed 256×256 face images
3. `model.state_dict` — existing weights for fine-tuning

**Anti-Idle Hack:** Before running Cell 4, open browser DevTools (F12) → Console → paste:
```javascript
function KeepClicking(){
  console.log('Keeping Colab alive...');
  document.querySelector('colab-connect-button').shadowRoot.getElementById('connect').click();
}
setInterval(KeepClicking, 60000);
```

In [ ]:
# Cell 1: Mount Google Drive & Install Dependencies
from google.colab import drive
drive.mount('/content/drive')

!pip install -q jsonlines tqdm "matplotlib<3.8" wandb

# Setup W&B for persistent cloud logging (survives session crashes)
import wandb
wandb.login()
print('Drive mounted + dependencies installed.')

In [ ]:
# Cell 2: Copy files from Drive to fast local storage & unzip
import os

# Copy training code
!cp /content/drive/MyDrive/inpaint_training.zip /content/
!unzip -qo /content/inpaint_training.zip -d /content/inpaint_project

# Copy and unzip preprocessed images to LOCAL disk (fast I/O)
!cp /content/drive/MyDrive/celeba_256.zip /content/
!mkdir -p /content/dataset/faces
!unzip -qo /content/celeba_256.zip -d /content/dataset/faces

# Copy base model for fine-tuning
!cp /content/drive/MyDrive/model.state_dict /content/

# Download QuickDraw mask data (correct ndjson vector format)
!mkdir -p /content/dataset/masks
!wget -q https://storage.googleapis.com/quickdraw_dataset/full/simplified/face.ndjson -P /content/dataset/masks

# Create checkpoint dir DIRECTLY on Google Drive (crash-proof)
!mkdir -p /content/drive/MyDrive/facerestore_checkpoints

# Verify everything
face_count = len(os.listdir('/content/dataset/faces'))
print(f'Face images: {face_count}')
print(f'Mask files: {os.listdir("/content/dataset/masks")}')
print(f'Model exists: {os.path.exists("/content/model.state_dict")}')
print(f'Train.py exists: {os.path.exists("/content/inpaint_project/train.py")}')
print(f'Drive checkpoint dir: {os.path.exists("/content/drive/MyDrive/facerestore_checkpoints")}')
print('\nAll ready!' if face_count > 1000 else '\nERROR: Not enough face images!')

In [ ]:
# Cell 3: Verify GPU
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU! Go to Runtime -> Change runtime type -> T4 GPU')

In [ ]:
import os

!cp /content/inpaint_project/train.py /content/inpaint_project/inpaint/
os.chdir('/content/inpaint_project/inpaint')

drive_checkpoint = '/content/drive/MyDrive/facerestore_checkpoints/last.pth'
original_model = '/content/model.state_dict'
resume_from = drive_checkpoint if os.path.exists(drive_checkpoint) else original_model
print(f'Auto-resume from: {resume_from}')

!python train.py \
  --train-dir /content/dataset/faces \
  --masks-dir /content/dataset/masks \
  --resume {resume_from} \
  --output-dir /content/drive/MyDrive/facerestore_checkpoints \
  --device cuda:0 \
  --batch-size 16 \
  --num-workers 2 \
  --epochs 20 \
  --lr 2e-5

In [ ]:
# Cell 5: Check what was saved (run after training or after a crash)
import os
checkpoint_dir = '/content/drive/MyDrive/facerestore_checkpoints'
if os.path.exists(checkpoint_dir):
    files = sorted(os.listdir(checkpoint_dir))
    print(f'Saved checkpoints ({len(files)} files):')
    for f in files:
        size_mb = os.path.getsize(os.path.join(checkpoint_dir, f)) / (1024*1024)
        print(f'  {f} ({size_mb:.1f} MB)')
else:
    print('No checkpoint directory found.')

## Resuming Training (Day 2, 3, etc.)

If Colab disconnected or you ran out of time:
1. Start a new Colab session with T4 GPU
2. Run Cells 1-3 (mount Drive, setup, verify GPU)
3. Run Cell 4 — it **automatically** detects your last checkpoint on Drive and resumes from it

No manual changes needed. The model picks up exactly where it left off.